In [158]:
import pandas as pd
import numpy as np
import random
from pomelo.utils import Hal

In [236]:
hal = Hal()

sql = """
select distinct pd_sold.year_week_sold
        ,pd_sold.year_month_id
        ,pd_sold.id_shop
        ,pd_sold.id_warehouse
        ,pd_sold.id_store
        ---- By week sold, month id, shop, warehouse, store, product launched  ----
        
        ---- total no. of style for exiting products  ----
        ,case when sum(exist_styles.active_product) is null then 0
            else sum(exist_styles.active_product) end as tot_product_existing
            
         ---- total no. of style for new arivals [1 weeks] ----    
        ,case when pd_launched.product_launched is null then 0 
            else pd_launched.product_launched end as tot_product_launched 
            
        -- total no of style
        ,tot_product_existing+tot_product_launched total_products        
        
        ---- sales unit new products  ----
        ,case when sum(pd_sold.net_units_sold) = 0 then 0
        when sum(pd_sold.new_pd_sold) <= 0 then 0
            else sum(pd_sold.new_pd_sold) 
            end as new_pd_sold
         
         ---- sales unit new products  ----
        ,case when sum(pd_sold.net_units_sold) = 0 then 0
        when sum(pd_sold.old_pd_sold) <= 0 then 0
            else sum(pd_sold.old_pd_sold) 
            end as old_pd_sold
            
        ---- ** PROPORTION NEW PRODUCT **  ----
        ,case when sum(pd_sold.net_units_sold) = 0 then 0
        when sum(pd_sold.new_pd_sold) <= 0 then 0
            else round(cast(sum(pd_sold.new_pd_sold) AS float)/cast(sum(pd_sold.net_units_sold) AS float),2) 
            end as prop_new_pd
        ---- ** PROPORTION OLD PRODUCT **  ----
        ,case when sum(pd_sold.net_units_sold) = 0 then 0
        when sum(pd_sold.old_pd_sold) <= 0 then 0
            else round(cast(sum(pd_sold.old_pd_sold) AS float)/cast(sum(pd_sold.net_units_sold) AS float),2) 
            end as prop_existing_pd
            

        from
        (----/* PRODUCT SOLD FOR EXISTING STYLES */----
        select distinct t.year_week_sold
        ,t.year_month_id
        ,t.id_shop
        ,t.id_warehouse
        ,t.id_store
        --- new & old - normal & lookbook ---
        ,case when product_age = 'new' then sum(t.tot_net_units_sold) else 0 end as new_pd_sold
        ,case when product_age = 'old' then sum(t.tot_net_units_sold) else 0 end as old_pd_sold

        ,new_pd_sold + old_pd_sold as net_units_sold
        from 
        (--- fact sales ---
        select dp2.id_product 
        ,fso2.id_shop
        --- case erply_store_id ---
        ,case when fso2.id_store = '5b989110a9afac06f24b80ce' then '231'
            when fso2.id_store = '5b55a681868e636a8ab1eaa2' then '261'
            when fso2.id_store = '5bb2e6ac1e5abf2252df039c' then '251'
            when fso2.id_store = '5bbedd0bc3cef17af5b1a3ab' then '11'
            when fso2.id_store = '5b9890f4d49bab0743f5689e' then '241'
            when fso2.id_store = '5a7c011b6878b592402d76bd' then '371'
            else fso2.id_store end as id_store
        ,case when id_shop = '2' and year_week_id < '202017' then '1'
            when id_shop = '2' and year_week_id >= '202017' then '11'
            when id_shop = '11' and year_week_id < '202038' then '1'
            when id_shop = '11' and year_week_id >= '202038' then '11'
            when id_shop = '10' then '1'
            when id_shop = '12' then '1'
            when id_shop = '4' then '1'
            when id_shop = '14' then '11'
            else id_shop end as id_warehouse
        ,ca.year_week_id as year_week_sold
        ,ca.year_month_id
        --- first date available ---
        ,min(pd_avl.year_week_available) as year_week_avl
        
        --- product age ---
        
        ,case when year_week_avl = ca.year_week_id then 'new'
        when ca.year_week_id - year_week_avl = 1 then 'new' 
        when ca.year_week_id - year_week_avl = 2 then 'new' 
        else 'old' end as product_age
        ,coalesce (sum(CASE WHEN fso2.transaction_type ='Return' THEN -fso2.product_quantity ELSE fso2.product_quantity END),0) as tot_net_units_sold
        from dwh.fact_sales_offline fso2
        ---/* LOOKBOOK COLLECTION */---
        left join (select distinct id_product
        ,sku_complete
        ,date_released
        from dwh.dim_product
        where date_released between date('2016-12-26') and date(getdate()-1)
        and id_product is not null
        and original_price_th is not NULL 
        and date_released is not NULL 
        and parent_product_line is not null
        and product_name is not null
        and product_line is not null
        and original_price_th != 0
        and henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
        and parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
        and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
        and id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
        and active = 1
        ) dp2
        on fso2.id_product = dp2.id_product
        ----/* Available date from NS */----
        left join (select sku_complete
        ,to_location_name
        ,case when to_id = 14 then 11
            when to_id = 29 then 211
            when to_id = 31 then 221
            when to_id = 69 then 231
            when to_id = 70 then 241
            when to_id = 71 then 251
            when to_id = 72 then 261
            when to_id = 73 then 281
            when to_id = 81 then 12
            when to_id = 92 then 311
            when to_id = 93 then 341
            when to_id = 95 then 321
            when to_id = 96 then 291
            when to_id = 99 then 301
            when to_id = 102 then 361
            when to_id = 103 then 331
            when to_id = 115 then 351
            when to_id = 116 then 381
            when to_id = 130 then 15
            when to_id = 131 then 371
            when to_id = 132 then 32
            when to_id = 159 then 391
            when to_id = 163 then 35
            when to_id = 165 then 42
            when to_id = 168 then 401
            when to_id = 175 then 411
            when to_id = 177 then 431
            when to_id = 178 then 111
            when to_id = 182 then 391
            when to_id = 186 then 441
            else null
            end as id_store
        ,min(ca.year_week_id) as year_week_available
        ,min(date(date_received)) as first_available_date
        FROM dwh.fact_ns_transfer_order ns
        ----/* YEAR WEEK ID FOR FIRST AVAILABLE DATE  */----
        left join (select full_date
        , case when ca.week_of_year_number < 10 then concat(convert(varchar(4),ca.year_id),right('0'+cast(ca.week_of_year_number AS varchar(2)),2))
            when ca.week_of_year_number = 53 then concat(convert(varchar(4),(ca.year_id+1)),'01')
            else concat(ca.year_id,ca.week_of_year_number) end as year_week_id
        from dwh.dim_calendar ca) ca
        on  date(ns.date_received) = ca.full_date 
        where id_store is not null
        GROUP BY 1,2,3
        order by sku_complete
        ) pd_avl
        on dp2.sku_complete = pd_avl.sku_complete and pd_avl.id_store = fso2.id_store 
        ----/* YEAR WEEK ID FOR TRANSACTION  */----
        left join (select full_date
        ,case when ca.month_id < 10
            then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.month_id AS VARCHAR(2)),2))
            else concat(ca.year_id,ca.week_of_year_number) end as year_month_id
        , case when ca.week_of_year_number < 10
            then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.week_of_year_number AS VARCHAR(2)),2))
            when ca.week_of_year_number = 53
            then concat(convert(varchar(4),(ca.year_id+1)),'01')
            else concat(ca.year_id,ca.week_of_year_number) end as year_week_id
        from dwh.dim_calendar ca) ca
        on  date(fso2.transaction_time) = ca.full_date
        where fso2.revenue_usd >0
        ---- NOTE: Marketing spend started 2019 only ----
        --and transaction_time >= date('2019-01-01')
        and sales_id is not null
        and dp2.id_product is not null
        and pd_avl.year_week_available is not null
        group by dp2.id_product
            ,fso2.id_shop
            ,fso2.id_store
            ,ca.year_week_id
            ,ca.year_month_id
        having year_week_sold >= year_week_avl) t
        group by t.year_week_sold
            ,t.id_shop
            ,t.year_month_id
            ,t.id_warehouse
            ,t.id_store
            ,product_age) pd_sold
        ----/* EXISTING PRODUCT */----
        ---- get count total active product ----
        ---- group by shop, store, week, lookbook ----
        left join 
        (select ca.year_week_id
        ,case when fisom.id_shop is null then '1' else fisom.id_shop end as id_shop
        ,case when id_store = '5b989110a9afac06f24b80ce' then '231'
            when id_store = '5b55a681868e636a8ab1eaa2' then '261'
            when id_store = '5bb2e6ac1e5abf2252df039c' then '251'
            when id_store = '5bbedd0bc3cef17af5b1a3ab' then '11'
            when id_store = '5b9890f4d49bab0743f5689e' then '241'
            when id_store = '5a7c011b6878b592402d76bd' then '371'
            else id_store
            end as id_store
        ,count(distinct dp2.id_product) as active_product
        from dwh.fact_inventory_snapshot_offline_master fisom 
        left join (select distinct id_product
        from dwh.dim_product dp
        where dp.date_released between date('2016-12-26') and date(getdate()-1)
        and dp.id_product is not null
        and dp.original_price_th is not NULL 
        and dp.date_released is not NULL 
        and dp.parent_product_line is not null
        and dp.product_name is not null
        and dp.product_line is not null
        and dp.original_price_th != 0
        and dp.henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
        --,'Lingerie Bodysuit', 'Sportswear Bottoms', 'Sportswear Outerwear', 'Sportswear Tops', 'Swimwear')
        and dp.parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
        and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
        and dp.id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
        and dp.active = 1) dp2
        on fisom.id_product = dp2.id_product	
        left join (select full_date
        , case when ca.week_of_year_number < 10
        then concat(convert(varchar(4),ca.year_id),right('0'+cast(ca.week_of_year_number AS varchar(2)),2))
        when ca.week_of_year_number = 53
        then concat(convert(varchar(4),(ca.year_id+1)),'01')
        else concat(ca.year_id,ca.week_of_year_number) 
        end as year_week_id
        from dwh.dim_calendar ca) ca
        on fisom.snapshot_date = ca.full_date
        where active = 1
        and fisom.snapshot_date between date('2018-06-22') and date(getdate()-1)
        and id_store not in ('1','201', '25', '271')	
        group by id_shop
            ,id_store
            ,ca.year_week_id) exist_styles
        on pd_sold.year_week_sold = exist_styles.year_week_id and pd_sold.id_store = exist_styles.id_store
        ----/* NEW PRODUCT LAUNCHED */----
        ---- get total product launched in that week ----
        left join (select distinct year_week_available
        ,id_store
        ,sum(tot_product_launched) as product_launched 
        from (select distinct avl_store.year_week_available
        ,avl_store.id_store
        ,count(distinct avl_store.id_product) as tot_product_launched
        from (select dp2.id_product
        ,dp2.sku_complete 
        ,to_location_name
        ,case when to_id = 14 then 11
            when to_id = 29 then 211
            when to_id = 31 then 221
            when to_id = 69 then 231
            when to_id = 70 then 241
            when to_id = 71 then 251
            when to_id = 72 then 261
            when to_id = 73 then 281
            when to_id = 81 then 12
            when to_id = 92 then 311
            when to_id = 93 then 341
            when to_id = 95 then 321
            when to_id = 96 then 291
            when to_id = 99 then 301
            when to_id = 102 then 361
            when to_id = 103 then 331
            when to_id = 115 then 351
            when to_id = 116 then 381
            when to_id = 130 then 15
            when to_id = 131 then 371
            when to_id = 132 then 32
            when to_id = 159 then 391
            when to_id = 163 then 35
            when to_id = 165 then 42
            when to_id = 168 then 401
            when to_id = 175 then 411
            when to_id = 177 then 431
            when to_id = 178 then 111
            when to_id = 182 then 391
            when to_id = 186 then 441
            else null
            end as id_store
        ,min(ca.year_week_id) as year_week_available
        ,min(date(date_received)) as first_available_date
        from dwh.fact_ns_transfer_order ns
        left join (select full_date
        ,case when ca.week_of_year_number < 10
            then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.week_of_year_number AS VARCHAR(2)),2))
            when ca.week_of_year_number = 53
            then concat(convert(varchar(4),(ca.year_id+1)),'01')
            else concat(ca.year_id,ca.week_of_year_number) end as year_week_id
        from dwh.dim_calendar ca) ca
        on  date(ns.date_received) = ca.full_date 
        left join (select id_product
        ,sku_complete 
        from dwh.dim_product dp
        where dp.date_released between date('2016-12-26') and date(getdate()-1)
            and dp.id_product is not null
            and dp.original_price_th is not NULL 
            and dp.date_released is not NULL 
            and dp.parent_product_line is not null
            and dp.product_name is not null
            and dp.product_line is not null
            and dp.original_price_th != 0
            and dp.henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
            and dp.parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
            and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
            and dp.id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
            and dp.active = 1) dp2
        on ns.sku_complete = dp2.sku_complete 
        where id_store is not null
            and id_product is not null
        GROUP BY 1,2,3,4) avl_store
        where year_week_available is not null 
        and id_store is not null
        group by avl_store.year_week_available
            ,avl_store.id_store)
        group by 1,2) pd_launched
        on pd_sold.year_week_sold = pd_launched.year_week_available and pd_sold.id_store = pd_launched.id_store
        where active_product is not null
        group by pd_sold.year_week_sold
            ,pd_sold.year_month_id
            ,pd_sold.id_shop
            ,pd_sold.id_warehouse
            ,pd_sold.id_store
            ,pd_launched.product_launched
        order by pd_sold.year_week_sold
"""

In [237]:
df = hal.get_pandas_df(sql)

In [238]:
df

,year_week_sold,year_month_id,id_shop,id_warehouse,id_store,tot_product_existing,tot_product_launched,total_products,new_pd_sold,old_pd_sold,prop_new_pd,prop_existing_pd
0,201852,201852,1,1,231,18,2,20,1,0,1.00,0.00
1,201902,201901,1,1,261,50,9,59,131,0,1.00,0.00
2,201902,201901,1,1,251,58,6,64,94,0,1.00,0.00
3,201902,201901,1,1,241,16,8,24,5,0,1.00,0.00
4,201903,201901,1,1,241,16,6,22,29,0,1.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
2006,202136,202109,1,1,401,378,40,418,194,286,0.40,0.60
2007,202136,202109,11,11,111,314,0,314,9,33,0.21,0.79
2008,202136,202109,1,1,301,896,0,896,42,590,0.07,0.93
2009,202136,202109,1,1,221,330,67,397,880,696,0.56,0.44


In [239]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2011 entries, 0 to 2010
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   year_week_sold        2011 non-null   object 
 1   year_month_id         2011 non-null   object 
 2   id_shop               2011 non-null   object 
 3   id_warehouse          2011 non-null   object 
 4   id_store              2011 non-null   object 
 5   tot_product_existing  2011 non-null   int64  
 6   tot_product_launched  2011 non-null   int64  
 7   total_products        2011 non-null   int64  
 8   new_pd_sold           2011 non-null   int64  
 9   old_pd_sold           2011 non-null   int64  
 10  prop_new_pd           2011 non-null   float64
 11  prop_existing_pd      2011 non-null   float64
dtypes: float64(2), int64(5), object(5)
memory usage: 188.7+ KB


In [240]:
df["tot_product_launched"] = df["tot_product_launched"].astype(float)
df["tot_product_existing"] = df["tot_product_existing"].astype(float)
df["total_products"] = df["total_products"].astype(float)

In [241]:
df["prop_new_styles"] = np.round(df["tot_product_launched"] / df["total_products"], 2)
df["prop_old_styles"] = np.round(df["tot_product_existing"] / df["total_products"], 2)

In [242]:
df["id_shop"].value_counts()

1     1723
2      207
5       72
11       9
Name: id_shop, dtype: int64

In [243]:
df["year_week_sold"].max()

'202136'

In [244]:
df["year_week_sold"].min()

'201852'

In [245]:
len(df["year_week_sold"].value_counts())

136

In [246]:
df.rename(
    columns={
        "total_products": "tot_products",
        "prop_new_pd": "prop_new_sold",
        "prop_existing_pd": "prop_existing_sold",
        "prop_old_styles": "prop_existing_styles",
    },
    inplace=True,
)

In [247]:
df_sum = (
    df.groupby(["year_week_sold", "id_shop"])[
        "tot_product_existing",
        "tot_product_launched",
        "tot_products",
        "new_pd_sold",
        "old_pd_sold",
    ]
    .sum()
    .reset_index()
)
df_sum["tot_pd_sold"] = df_sum["new_pd_sold"] + df_sum["old_pd_sold"]

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:1: FutureWarning:

Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.



In [248]:
df_sum["prop_new_styles"] = np.round(
    df_sum["tot_product_launched"] / df_sum["tot_products"], 2
)
df_sum["prop_existing_styles"] = np.round(
    df_sum["tot_product_existing"] / df_sum["tot_products"], 2
)
df_sum["prop_new_sold"] = np.round(df_sum["new_pd_sold"] / df_sum["tot_pd_sold"], 2)
df_sum["prop_existing_sold"] = np.round(
    df_sum["old_pd_sold"] / df_sum["tot_pd_sold"], 2
)

In [221]:
# 1 weeks
df_sum.groupby(["id_shop"])[
    "prop_new_sold", "prop_existing_sold", "prop_new_styles", "prop_existing_styles"
].mean().reset_index().style.format(
    {
        "prop_new_sold": "{:.2%}",
        "prop_existing_sold": "{:.2%}",
        "prop_new_styles": "{:.2%}",
        "prop_existing_styles": "{:.2%}",
    }
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:2: FutureWarning:

Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.



,id_shop,prop_new_sold,prop_existing_sold,prop_new_styles,prop_existing_styles
0,1,19.00%,81.00%,6.81%,93.19%
1,11,1.00%,99.00%,1.62%,98.38%
2,2,11.12%,88.88%,6.42%,93.58%
3,5,9.49%,90.51%,5.91%,94.09%


In [235]:
# 2 weeks
df_sum.groupby(["id_shop"])[
    "prop_new_sold", "prop_existing_sold", "prop_new_styles", "prop_existing_styles"
].mean().reset_index().style.format(
    {
        "prop_new_sold": "{:.2%}",
        "prop_existing_sold": "{:.2%}",
        "prop_new_styles": "{:.2%}",
        "prop_existing_styles": "{:.2%}",
    }
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:2: FutureWarning:

Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.



,id_shop,prop_new_sold,prop_existing_sold,prop_new_styles,prop_existing_styles
0,1,39.36%,60.64%,6.75%,93.25%
1,11,18.25%,81.75%,1.62%,98.38%
2,2,25.85%,74.15%,6.27%,93.73%
3,5,27.09%,72.91%,5.86%,94.14%


In [249]:
# 3 weeks
df_sum.groupby(["id_shop"])[
    "prop_new_sold", "prop_existing_sold", "prop_new_styles", "prop_existing_styles"
].mean().reset_index().style.format(
    {
        "prop_new_sold": "{:.2%}",
        "prop_existing_sold": "{:.2%}",
        "prop_new_styles": "{:.2%}",
        "prop_existing_styles": "{:.2%}",
    }
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:2: FutureWarning:

Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.



,id_shop,prop_new_sold,prop_existing_sold,prop_new_styles,prop_existing_styles
0,1,52.09%,47.91%,6.83%,93.17%
1,11,34.00%,66.00%,1.75%,98.25%
2,2,36.96%,63.04%,6.26%,93.74%
3,5,40.89%,59.11%,6.09%,93.91%


In [155]:
df1 = (
    df.groupby(["id_shop"])[
        "prop_new_sold", "prop_existing_sold", "prob_new_styles", "prob_existing_styles"
    ]
    .mean()
    .reset_index()
)
df1.style.format(
    {
        "prop_new_sold": "{:.2%}",
        "prop_existing_sold": "{:.2%}",
        "prob_new_styles": "{:.2%}",
        "prob_existing_styles": "{:.2%}",
    }
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:1: FutureWarning:

Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.



,id_shop,prop_new_sold,prop_existing_sold,prob_new_styles,prob_existing_styles
0,1,49.54%,50.01%,6.94%,93.06%
1,11,29.00%,59.00%,1.56%,98.44%
2,2,37.81%,62.19%,5.65%,94.35%
3,5,37.64%,62.36%,5.87%,94.12%


In [142]:
df1 = (
    df.groupby(["id_shop"])[
        "prop_new_sold", "prop_existing_sold", "prob_new_styles", "prob_existing_styles"
    ]
    .mean()
    .reset_index()
)
df1.style.format(
    {
        "prop_new_sold": "{:.2%}",
        "prop_existing_sold": "{:.2%}",
        "prob_new_styles": "{:.2%}",
        "prob_existing_styles": "{:.2%}",
    }
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:1: FutureWarning:

Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.



,id_shop,prop_new_sold,prop_existing_sold,prob_new_styles,prob_existing_styles
0,1,36.05%,63.50%,6.80%,93.20%
1,11,16.22%,72.67%,1.44%,98.56%
2,2,25.61%,74.39%,5.58%,94.42%
3,5,24.61%,75.39%,5.74%,94.26%


In [124]:
df1 = (
    df.groupby(["id_shop"])[
        "prop_new_sold", "prop_existing_sold", "prob_new_styles", "prob_existing_styles"
    ]
    .mean()
    .reset_index()
)
df1.style.format(
    {
        "prop_new_sold": "{:.2%}",
        "prop_existing_sold": "{:.2%}",
        "prob_new_styles": "{:.2%}",
        "prob_existing_styles": "{:.2%}",
    }
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:1: FutureWarning:

Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.



,id_shop,prop_new_sold,prop_existing_sold,prob_new_styles,prob_existing_styles
0,1,16.76%,82.77%,6.93%,93.07%
1,11,0.89%,88.00%,1.44%,98.56%
2,2,10.94%,89.06%,5.80%,94.20%
3,5,8.75%,91.26%,5.96%,94.04%


In [106]:
df2 = (
    df.groupby(["id_shop", "id_store"])[
        "prop_new_sold", "prop_existing_sold", "prob_new_styles", "prob_existing_styles"
    ]
    .mean()
    .reset_index()
)
df2.style.format(
    {
        "prop_new_sold": "{:.2%}",
        "prop_existing_sold": "{:.2%}",
        "prob_new_styles": "{:.2%}",
        "prob_existing_styles": "{:.2%}",
    }
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:1: FutureWarning:

Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.



,id_shop,id_store,prop_new_sold,prop_existing_sold,prob_new_styles,prob_existing_styles
0,1,11,16.30%,83.70%,5.67%,94.33%
1,1,211,17.45%,82.55%,6.27%,93.73%
2,1,221,20.17%,79.83%,7.23%,92.77%
3,1,231,19.72%,80.28%,10.93%,89.07%
4,1,241,28.74%,71.27%,13.13%,86.87%
5,1,251,18.58%,81.43%,6.24%,93.76%
6,1,261,21.38%,78.62%,7.47%,92.53%
7,1,281,24.30%,57.52%,16.55%,83.45%
8,1,291,30.35%,69.65%,12.29%,87.71%
9,1,301,5.40%,94.60%,3.65%,96.35%


In [126]:
df3 = (
    df.groupby(["year_week_sold", "id_shop", "id_store"])[
        "prop_new_sold", "prop_existing_sold", "prob_new_styles", "prob_existing_styles"
    ]
    .mean()
    .reset_index()
)
# df3.style.format({'prop_new_sold': "{:.2%}",'prop_existing_sold': "{:.2%}",'prob_new_styles': "{:.2%}",'prob_existing_styles': "{:.2%}"})
# df3

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:1: FutureWarning:

Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.



In [129]:
import plotly.graph_objects as go

df_plot = df3[df3["id_shop"] == "1"]

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=("TH", "MY", "ID", "SG"),
)
fig = go.Figure(
    data=[
        go.Bar(
            name="prop_new_sold",
            x=df_plot["year_week_sold"],
            y=df_plot["prop_new_sold"],
        ),
        go.Bar(
            name="prop_existing_sold",
            x=df_plot["year_week_sold"],
            y=df_plot["prop_existing_sold"],
        ),
    ]
)
# Change the bar mode
fig.update_layout(barmode="stack")


fig.write_html(
    f"/home/ec2-user/SageMaker/business-intelligence-notebooks/dfm_clothing/offline_dfm_v2/plot_total_products.html"
)

ValueError: 
    Invalid element(s) received for the 'data' property of 
        Invalid elements include: [Figure({
    'data': [{'name': 'prop_new_sold',
              'type': 'bar',
              'x': [201852, 201902, 201902, ..., 202136, 202136, 202136],
              'y': array([1.  , 1.  , 1.  , ..., 0.29, 0.45, 0.  ])},
             {'name': 'prop_existing_sold',
              'type': 'bar',
              'x': [201852, 201902, 201902, ..., 202136, 202136, 202136],
              'y': array([0.  , 0.  , 0.  , ..., 0.71, 0.55, 1.  ])}],
    'layout': {'template': '...'}
})]

    The 'data' property is a tuple of trace instances
    that may be specified as:
      - A list or tuple of trace instances
        (e.g. [Scatter(...), Bar(...)])
      - A single trace instance
        (e.g. Scatter(...), Bar(...), etc.)
      - A list or tuple of dicts of string/value properties where:
        - The 'type' property specifies the trace type
            One of: ['bar', 'barpolar', 'box', 'c

In [123]:
# df3.to_csv('new_old_prob.csv',index=False)

In [66]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=("TH", "MY", "ID", "SG"),
)

fig.add_trace(
    go.Scatter(
        x=df[df["id_shop"] == "2"]["year_week_sold"],
        y=df[df["id_shop"] == "2"]["total_products"],
    ),
    row=4,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=df[df["id_shop"] == "5"]["year_week_sold"],
        y=df[df["id_shop"] == "5"]["total_products"],
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=df[df["id_shop"] == "11"]["year_week_sold"],
        y=df[df["id_shop"] == "11"]["total_products"],
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=df[df["id_shop"] == "1"]["year_week_sold"],
        y=df[df["id_shop"] == "1"]["total_products"],
    ),
    row=1,
    col=1,
)

fig.update_layout(height=1200, width=1500, title_text="week dist by warehouse")

fig.write_html(
    f"/home/ec2-user/SageMaker/business-intelligence-notebooks/dfm_clothing/offline_dfm_v2/plot_total_products.html"
)